# Agente Bancário com GraphRAG — Demonstração da API

Este notebook demonstra os **três modos de agente** do pipeline RAG bancário em produção na AWS (sa-east-1).

| Modo | Perspectiva | Requisito | Caso de uso |
|---|---|---|---|
| `segmento` | 3ª pessoa — gerente fala *sobre* o cliente | `dados_cliente` (6 features) | Análise de perfil e recomendação gerencial |
| `persona` | 1ª pessoa — arquétipo do segmento | `cluster_id` OU `dados_cliente` | Simulação de reação de cliente típico |
| `twin` | 1ª pessoa — o próprio cliente | `cliente_id` + `cluster_id` | Digital twin para teste A/B e personalização extrema |

Cada resposta inclui um **índice de confiabilidade** que mede o quanto o LLM utilizou o contexto RAG recuperado.

## Arquitetura RAG (3 camadas)

```
Pergunta
  │
  ├─ 1. BM25 OpenSearch  (t3.small, sa-east-1)
  │      ├─ segmento/persona → índice clientes-segmento-{cluster_id}
  │      └─ twin            → índice clientes-digital-twins (filtro por cliente_id)
  │
  ├─ 2. Grafo Neptune live  (db.t3.medium, IAM auth)
  │      ├─ Segmento → Produtos recomendados
  │      ├─ Segmento → Persona arquétipo
  │      └─ Cliente  → Clientes similares k-NN (modo twin)
  │
  └─ 3. Bedrock Claude  (global.anthropic.claude-sonnet-4-5, sa-east-1)
```

**Fluxo assíncrono:** `POST /query` → SQS → Worker Lambda → DynamoDB → `GET /status/{id}`

In [ ]:
import json
import time
import requests
from IPython.display import display, HTML, Markdown

API_URL = "https://3vqrahlmyg.execute-api.sa-east-1.amazonaws.com"

# ── Cores para o índice de confiabilidade ──────────────────────────────────────
NIVEL_COR = {"alto": "#2d7a2d", "medio": "#c47a00", "baixo": "#c0392b"}
NIVEL_BG  = {"alto": "#eafaea", "medio": "#fff8e1", "baixo": "#fdecea"}

def consultar(payload: dict, poll_interval: int = 3, timeout: int = 90) -> dict:
    """Envia query e aguarda resultado (polling)."""
    r = requests.post(f"{API_URL}/query", json=payload, timeout=30)
    r.raise_for_status()
    request_id = r.json()["request_id"]
    print(f"  request_id: {request_id}")

    for _ in range(timeout // poll_interval):
        time.sleep(poll_interval)
        status = requests.get(f"{API_URL}/status/{request_id}", timeout=10).json()
        st = status["status"]
        if st == "COMPLETED":
            resultado = json.loads(status["resultado"])
            resultado["_request_id"] = request_id
            return resultado
        if st == "FAILED":
            raise RuntimeError(f"Worker falhou:\n{status.get('erro', '')}")
        print(f"  {st}...", end="\r")

    raise TimeoutError(f"Sem resposta em {timeout}s (request_id={request_id})")


def exibir(resultado: dict) -> None:
    """Renderiza resposta + índice de confiabilidade em HTML."""
    modo       = resultado.get("modo", "")
    resposta   = resultado.get("resposta", "")
    ic         = resultado.get("indice_confiabilidade", {})
    score      = ic.get("score", 0)
    nivel      = ic.get("nivel", "baixo")
    fontes     = ic.get("fontes_ativas", [])
    cobertura  = ic.get("detalhes", {}).get("cobertura_fontes", 0)
    overlap    = ic.get("detalhes", {}).get("sobreposicao_lexica", 0)

    cor = NIVEL_COR[nivel]
    bg  = NIVEL_BG[nivel]

    fontes_html = "".join(
        f'<span style="background:#dde;border-radius:4px;padding:2px 7px;'
        f'font-size:0.82em;margin-right:4px">{f}</span>'
        for f in fontes
    ) or '<span style="color:#999">nenhuma</span>'

    barra_largura = int(score * 200)

    html = f"""
    <div style="font-family:sans-serif;max-width:860px;border:1px solid #ddd;
                border-radius:8px;overflow:hidden;margin:12px 0">

      <div style="background:#2c3e50;color:#ecf0f1;padding:10px 16px;
                  font-size:0.85em;letter-spacing:.5px">
        MODO: <strong>{modo.upper()}</strong>
        {'&nbsp;·&nbsp;' + resultado.get('segmento','') if resultado.get('segmento') else ''}
        {'&nbsp;·&nbsp;' + resultado.get('persona_nome','') if resultado.get('persona_nome') else ''}
        {'&nbsp;·&nbsp;' + resultado.get('cliente_id','') if resultado.get('cliente_id') else ''}
      </div>

      <div style="padding:16px 18px;white-space:pre-wrap;line-height:1.6;
                  font-size:0.93em;border-bottom:1px solid #eee">{resposta}</div>

      <div style="background:{bg};padding:10px 18px">
        <div style="display:flex;align-items:center;gap:16px;flex-wrap:wrap">
          <div>
            <span style="font-size:1.5em;font-weight:700;color:{cor}">{score:.2f}</span>
            <span style="margin-left:6px;background:{cor};color:white;border-radius:4px;
                         padding:2px 8px;font-size:0.78em;font-weight:600">{nivel.upper()}</span>
          </div>
          <div style="flex:1;min-width:160px">
            <div style="background:#ddd;border-radius:4px;height:8px">
              <div style="background:{cor};width:{barra_largura}px;max-width:200px;
                           height:8px;border-radius:4px"></div>
            </div>
          </div>
          <div style="font-size:0.82em;color:#555">
            <strong>Fontes:</strong> {fontes_html}<br>
            <strong>Cobertura:</strong> {cobertura:.2f} &nbsp;
            <strong>Sobreposição léxica:</strong> {overlap:.2f}
          </div>
        </div>
      </div>
    </div>
    """
    display(HTML(html))

print("Setup concluído. API:", API_URL)

---
## Modo `segmento` — Gerente analisa o cliente

O LLM recebe o **perfil do segmento K-Means** (recuperado do OpenSearch) e os **produtos e persona do grafo Neptune**.
Responde em **3ª pessoa**, como um gerente de relacionamento.

**Cliente:** Premium Conservador · renda R$ 18.000 · score 830 · 52 anos

In [ ]:
print("Enviando consulta modo segmento...")

resultado_segmento = consultar({
    "cliente_id": "C001",
    "dados_cliente": {
        "idade": 52,
        "renda_mensal": 18000,
        "saldo_medio": 140000,
        "transacoes_mes": 8,
        "score_credito": 830,
        "num_produtos": 5
    },
    "pergunta": "Que produto de investimento faz mais sentido para este cliente agora?",
    "modo": "segmento"
})

exibir(resultado_segmento)

---
## Modo `persona` — Arquétipo do segmento responde em 1ª pessoa

O LLM encarna a **persona arquétipo do cluster** (Júlia — Jovem Digital) e responde como esse perfil típico reagiria.
Ideal para testar framing de ofertas e linguagem de comunicação.

**Cluster 1:** Jovem Digital · Júlia · designer freelancer · 26 anos · cashback · app

In [ ]:
print("Enviando consulta modo persona...")

resultado_persona = consultar({
    "cluster_id": 1,
    "pergunta": "O banco lançou um cartão com cashback de 3% e aprovação pelo app. Você toparia?",
    "modo": "persona"
})

exibir(resultado_persona)

---
## Modo `twin` — Digital twin do cliente real

O LLM simula **exatamente** como o cliente `GRAPH-C00-0001` — com seus dados individuais —
pensaria e reagiria. Usa:
- **BM25:** perfil individual indexado no OpenSearch
- **Neptune:** produtos do segmento + 3 clientes mais similares (k-NN)

**Cliente:** GRAPH-C00-0001 · Premium Conservador · 43 anos · renda R$ 4.685 · score 720

In [ ]:
print("Enviando consulta modo twin...")

resultado_twin = consultar({
    "cliente_id": "GRAPH-C00-0001",
    "cluster_id": 0,
    "dados_cliente": {
        "idade": 43,
        "renda_mensal": 4685,
        "saldo_medio": 35000,
        "transacoes_mes": 6,
        "score_credito": 720,
        "num_produtos": 3
    },
    "pergunta": "Considerando meu perfil e clientes parecidos comigo, que estratégia de previdência faz sentido?",
    "modo": "twin"
})

exibir(resultado_twin)

---
## Variação entre segmentos — Mesma pergunta, 4 perfis

Demonstra como o mesmo produto é percebido diferentemente por cada arquétipo de cliente.

| cluster | Segmento | Persona |
|---|---|---|
| 0 | Premium Conservador | Carlos, gerente, 52a |
| 1 | Jovem Digital | Júlia, freelancer, 26a |
| 2 | Alto Risco | Roberto, autônomo, 43a |
| 3 | Massa Estável | Ana, funcionária pública, 38a |

In [ ]:
PERGUNTA = "O banco oferece um empréstimo pessoal de R$ 5.000 com taxa de 2,5% ao mês. Você toparia?"

resultados_persona = {}
for cluster_id in range(4):
    print(f"  Consultando cluster {cluster_id}...")
    resultados_persona[cluster_id] = consultar({
        "cluster_id": cluster_id,
        "pergunta": PERGUNTA,
        "modo": "persona"
    })

print("\nRespostas:")
for cluster_id, res in resultados_persona.items():
    exibir(res)

---
## Comparativo: índice de confiabilidade por modo

Consolida os resultados das consultas anteriores numa tabela comparativa.

In [ ]:
def tabela_comparativa(resultados: list) -> None:
    linhas = ""
    for r in resultados:
        modo      = r.get("modo", "")
        segmento  = r.get("segmento") or r.get("persona_nome") or r.get("cliente_id", "")
        ic        = r.get("indice_confiabilidade", {})
        score     = ic.get("score", 0)
        nivel     = ic.get("nivel", "")
        fontes    = ", ".join(ic.get("fontes_ativas", [])) or "—"
        cobertura = ic.get("detalhes", {}).get("cobertura_fontes", 0)
        overlap   = ic.get("detalhes", {}).get("sobreposicao_lexica", 0)
        cor       = NIVEL_COR[nivel]
        bg        = NIVEL_BG[nivel]
        barra     = int(score * 120)

        linhas += f"""
        <tr style="background:{bg}">
          <td style="padding:8px 12px"><code>{modo}</code></td>
          <td style="padding:8px 12px">{segmento}</td>
          <td style="padding:8px 12px;text-align:center">
            <strong style="color:{cor}">{score:.2f}</strong>
            <div style="background:#ddd;border-radius:3px;height:6px;margin-top:3px">
              <div style="background:{cor};width:{barra}px;height:6px;border-radius:3px"></div>
            </div>
          </td>
          <td style="padding:8px 12px;text-align:center">
            <span style="background:{cor};color:white;border-radius:4px;
                         padding:2px 8px;font-size:0.8em">{nivel.upper()}</span>
          </td>
          <td style="padding:8px 12px;font-size:0.82em">{fontes}</td>
          <td style="padding:8px 12px;text-align:center">{cobertura:.2f}</td>
          <td style="padding:8px 12px;text-align:center">{overlap:.2f}</td>
        </tr>"""

    html = f"""
    <table style="border-collapse:collapse;width:100%;font-family:sans-serif;font-size:0.9em">
      <thead>
        <tr style="background:#2c3e50;color:white">
          <th style="padding:10px 12px;text-align:left">Modo</th>
          <th style="padding:10px 12px;text-align:left">Segmento / Persona</th>
          <th style="padding:10px 12px;text-align:center">Score</th>
          <th style="padding:10px 12px;text-align:center">Nível</th>
          <th style="padding:10px 12px;text-align:left">Fontes RAG ativas</th>
          <th style="padding:10px 12px;text-align:center">Cobertura</th>
          <th style="padding:10px 12px;text-align:center">Sobreposição</th>
        </tr>
      </thead>
      <tbody>{linhas}</tbody>
    </table>"""
    display(HTML(html))


# Consolida resultados das seções anteriores
todos = [
    resultado_segmento,
    resultado_persona,
    resultado_twin,
]

tabela_comparativa(todos)

---
## Como interpretar o índice de confiabilidade

O índice é calculado em duas dimensões:

| Componente | Peso | Descrição |
|---|---|---|
| **Cobertura de fontes** | 85% | Quais camadas RAG entregaram conteúdo (BM25=0.40, Neptune live=0.30, Neptune sync=0.15) |
| **Sobreposição léxica** | 15% | Fração de termos-chave do contexto recuperado que aparece na resposta |

### Escala de score

| Score | Nível | Significado |
|---|---|---|
| ≥ 0.70 | **ALTO** | 2+ fontes RAG ativas; LLM usou o contexto de forma verificável |
| 0.40 – 0.69 | **MÉDIO** | 1 fonte ativa ou sobreposição léxica parcial |
| < 0.40 | **BAIXO** | Sem contexto recuperado — resposta baseada em conhecimento paramétrico |

> **Nota sobre o modo persona:** a sobreposição léxica tende a ser menor porque o LLM reescreve
> o contexto em linguagem coloquial de 1ª pessoa, mas a cobertura de fontes reflete corretamente
> que ambas as camadas foram consultadas.

---
## Demonstração de degradação — cliente sem dados indexados

Mostra o que acontece quando o cliente não está no índice OpenSearch:
- **Antes do fix:** GRAPH-C* não estava indexado → score baixo
- **Hoje:** GRAPH-C* indexados pelo `indexar_twins_graph.py`

Simulamos um cliente hipotético `DEMO-X999` que não existe no índice:

In [ ]:
print("Enviando consulta com cliente sem dados no índice (DEMO-X999)...")

resultado_sem_indice = consultar({
    "cliente_id": "DEMO-X999",
    "cluster_id": 1,
    "dados_cliente": {
        "idade": 28,
        "renda_mensal": 3200,
        "saldo_medio": 5000,
        "transacoes_mes": 20,
        "score_credito": 650,
        "num_produtos": 2
    },
    "pergunta": "Devo aceitar o limite de crédito pré-aprovado de R$ 3.000?",
    "modo": "twin"
})

exibir(resultado_sem_indice)

print("\n--- Comparativo ---")
tabela_comparativa([resultado_twin, resultado_sem_indice])

---
## Arquitetura GraphRAG — Diagrama de Interação

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
from plantuml_helper import show_plantuml

puml_path = os.path.join("decisoes_projeto", "graphrag_interaction.puml")
with open(puml_path, encoding="utf-8") as f:
    diagrama = f.read()

show_plantuml(diagrama, "GraphRAG Pipeline — sa-east-1")

---
## Consulta livre

Edite as variáveis abaixo para testar sua própria pergunta:

In [ ]:
# ── Edite aqui ─────────────────────────────────────────────────────────────────
MODO       = "segmento"          # "segmento" | "persona" | "twin"
PERGUNTA   = "Como este cliente se comporta em relação a investimentos de longo prazo?"
CLIENTE_ID = "C002"              # usado em modo twin
CLUSTER_ID = None                # None = classificar automaticamente pelos dados

DADOS_CLIENTE = {
    "idade":          35,
    "renda_mensal":   4500,
    "saldo_medio":    8000,
    "transacoes_mes": 15,
    "score_credito":  680,
    "num_produtos":   3,
}
# ───────────────────────────────────────────────────────────────────────────────

payload = {"pergunta": PERGUNTA, "modo": MODO, "dados_cliente": DADOS_CLIENTE}
if CLIENTE_ID:
    payload["cliente_id"] = CLIENTE_ID
if CLUSTER_ID is not None:
    payload["cluster_id"] = CLUSTER_ID

print(f"Consultando modo {MODO!r}...")
resultado_livre = consultar(payload)
exibir(resultado_livre)